# Chicago Crime Analysis & Patrol Optimization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist

import folium
from folium.plugins import HeatMap


## Load Data

In [ ]:
df = pd.read_csv("chicago_crime.csv")
df = df.drop(columns=["Case Number","IUCR","FBI Code"], errors='ignore')
df = df.dropna(subset=["Latitude","Longitude"])
df.head()

## Feature Engineering

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Hour"] = df["Date"].dt.hour
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Month"] = df["Date"].dt.month

features = ["Latitude","Longitude","Hour","DayOfWeek","Month"]
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## PCA

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

plt.plot(pca.explained_variance_ratio_.cumsum())
plt.title("PCA Variance")
plt.show()

## K-Means

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)

print("KMeans Silhouette:", silhouette_score(X_scaled, labels_k))

## DBSCAN Tuning

In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
nbrs = neighbors.fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)

distances = np.sort(distances[:,4])
plt.plot(distances)
plt.title("k-distance Graph")
plt.show()

## DBSCAN

In [ ]:
db = DBSCAN(eps=0.25, min_samples=5)
labels_db = db.fit_predict(X_scaled)

df["Cluster"] = labels_db

valid = labels_db != -1
if len(set(labels_db[valid])) > 1:
    print("DBSCAN Silhouette:", silhouette_score(X_scaled[valid], labels_db[valid]))

## Heatmap

In [ ]:
m = folium.Map(location=[df["Latitude"].mean(), df["Longitude"].mean()], zoom_start=11)
HeatMap(df[["Latitude","Longitude"]].values.tolist()).add_to(m)
m

## Cluster Map

In [ ]:
m2 = folium.Map(location=[df["Latitude"].mean(), df["Longitude"].mean()], zoom_start=11)

sample = df.sample(3000)

for _, row in sample.iterrows():
    color = "black" if row["Cluster"]==-1 else "red"
    folium.CircleMarker(location=[row["Latitude"],row["Longitude"]], radius=2, color=color).add_to(m2)

m2

## Patrol Optimization

In [ ]:
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

m3 = folium.Map(location=[df["Latitude"].mean(), df["Longitude"].mean()], zoom_start=11)

for pt in centroids:
    folium.Marker(location=[pt[0],pt[1]]).add_to(m3)

m3

## Distance Evaluation

In [ ]:
distances = cdist(X_scaled, kmeans.cluster_centers_)
print("Avg Distance:", distances.min(axis=1).mean())